**Step 3 of 5.** Consumes: `experimental-data/instances/toy/*.{json,mps}` (step 1 output, committed). Produces: `experimental-data/results/toy-hqc-mcms/{instance}_p{qaoa_depth}/iterations.jsonl` and `result.json` (gitignored). Runtime: worst-case ~40h (10 tasks × 4h each) if run sequentially; typical ~10-30h depending on convergence.

**Prerequisites**: Julia installation with `julia/MPS_JuliQAOA` project instantiated (see `julia/README.md`).

**Note:** This notebook runs the hybrid quantum-classical MCMS Benders decomposition with MPS-JuliQAOA (tensor network QAOA emulator) for the cut selection step. Results generate Figure 3 in the paper.

# Toy HQC-MCMS Benders Experiment

Runs hybrid quantum-classical MCMS Benders decomposition on 5 random toy VRP instances (5 customers, 3 vehicles) with QAOA-based cut selection at depths p={1, 3}.

**Total**: 10 tasks (5 instances × 2 QAOA depths)

**Solver configuration**:
- Master problem (MP): Cbc via PuLP
- Dual subproblems (DSP): HiGHS (returns extreme rays for feasibility cuts)
- Cut selection: **MPS-JuliQAOA** (QAOA tensor network emulator for minimum set cover QUBO)

This matches the "HQC-MCMS" configuration from Table 1 in the paper. The paper's Figure 3 plots only p=1 results; p=3 was explored alongside.

## 1. Imports and Helper Functions

In [1]:
import json
import time
import pulp
from pathlib import Path

from milp_engine.solvers.bender_milp_solver import BenderMILPSolver
from milp_engine.solvers.pulp_solver import PuLPSolver
from milp_engine.solvers.highs_solver import HiGHSSolver
from milp_engine.solvers.quantum_solver import QuantumSolver, JuliaMPSBackend
from milp_engine.criteria.exclusion_criterion import ExclusionCriterion
from milp_engine.strategies.minimum_set_cover_strategy import MinimumSetCoverStrategy

from notebooks.utils import BendersSolverWithTimeout

## 2. Configuration

Define experiment parameters inline (no YAML config files).

In [2]:
# Experiment parameters (from Table 1 in paper)
INSTANCES_DIR = Path("experimental-data/instances/toy")
RESULTS_DIR = Path("experimental-data/results/toy-hqc-mcms")

# Benders parameters
N_SUBPROBLEMS = 2 #5  # Fixed for toy instances
QAOA_DEPTH_VALUES = [1] #, 3]  # Parameter grid: QAOA depths (p in paper)
MAX_ITERATIONS = 250 # 1001
CONVERGENCE_BOUND = 0.01
TIME_BUDGET_SECONDS = 3600  # 1 hour per task

# Julia QAOA configuration (MPS tensor network emulator)
JULIA_EXECUTABLE = "julia"
JULIA_PROJECT_PATH = "julia/MPS_JuliQAOA"
QAOA_N_SHOTS = 1000
QAOA_MAXDIM = 16 # 64  # Bond dimension for MPS
QAOA_OPTIMIZER = "cobyla"
QAOA_OPTIMIZER_MAXITER = 1000
QAOA_RHOBEG = 1.5708  # π/2

# Complicating variable prefix (routing decisions)
Y_PREFIX = "x_"

print(f"Instances directory: {INSTANCES_DIR}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Parameter grid: qaoa_depth = {QAOA_DEPTH_VALUES}")
print(f"Time budget: {TIME_BUDGET_SECONDS}s ({TIME_BUDGET_SECONDS/3600:.1f}h) per task")
print(f"Total tasks: {len(list(INSTANCES_DIR.glob('*.mps'))) * len(QAOA_DEPTH_VALUES)}")

Instances directory: experimental-data/instances/toy
Results directory: experimental-data/results/toy-hqc-mcms
Parameter grid: qaoa_depth = [1]
Time budget: 3600s (1.0h) per task
Total tasks: 5


## 3. Run Experiments

Loop over all (instance, qaoa_depth) combinations. Each task is independent and writes to its own directory.

**Progress tracking**: Results are written to disk after each task completes, so you can interrupt and resume by skipping tasks that already have a `result.json` file.

**QAOA runtime**: MPS-JuliQAOA dominates the per-iteration time (~99% in paper results). Expect each iteration to take minutes, not seconds.

In [3]:
# Discover instances
instance_files = sorted(INSTANCES_DIR.glob("toy_*.mps"))
print(f"Found {len(instance_files)} toy instances\n")

# Build task list
tasks = []
for instance_path in instance_files[:2]:  #instance_files: # Limit to first 2 instances for testing
    for qaoa_depth in QAOA_DEPTH_VALUES:
        tasks.append((instance_path, qaoa_depth))

print(f"Total tasks: {len(tasks)}\n")
print("=" * 80)

Found 5 toy instances

Total tasks: 2



In [4]:
# Run tasks
for task_idx, (instance_path, qaoa_depth) in enumerate(tasks, 1):
    instance_name = instance_path.stem
    task_name = f"{instance_name}_p{qaoa_depth}"
    output_dir = RESULTS_DIR / task_name
    
    # Skip if already completed
    if (output_dir / "result.json").exists():
        print(f"[{task_idx}/{len(tasks)}] {task_name}: already completed — skipping")
        continue

    if not output_dir.exists():
        output_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\n[{task_idx}/{len(tasks)}] {task_name}")
    print(f"  Instance: {instance_name}")
    print(f"  QAOA depth (p): {qaoa_depth}")
    print(f"  Output: {output_dir}")
    
    try:
        # Load MILP from MPS file
        _, milp = pulp.LpProblem.fromMPS(instance_path)
        
        # Instantiate Julia QAOA backend
        julia_backend = JuliaMPSBackend(
            julia_options={
                "julia_executable": JULIA_EXECUTABLE,
                "julia_project_path": JULIA_PROJECT_PATH,
                "n_shots": QAOA_N_SHOTS,
                "reps": qaoa_depth,  # QAOA depth parameter
                "maxdim": QAOA_MAXDIM,
                "optimizer": QAOA_OPTIMIZER,
                "maxiter": QAOA_OPTIMIZER_MAXITER,
                "rhobeg": QAOA_RHOBEG,
                "cutoff": 1e-6,
            },
        )
        julia_backend.qaoa_log_dir = output_dir

        
        # Instantiate quantum solver for cut selection
        cut_selection_solver = QuantumSolver(
            backend=julia_backend,
            solver_options={"reps": qaoa_depth},  # redundant but kept for compatibility
        )
        
        # Instantiate Benders solver
        solver = BenderMILPSolver(
            criterion=ExclusionCriterion(),
            strategy=MinimumSetCoverStrategy(),
            mp_solver=PuLPSolver(),  # Cbc for master problem
            dsp_solver=HiGHSSolver(),  # HiGHS for dual subproblems
            cut_selection_solver=cut_selection_solver,  # QAOA for cut selection
            max_iterations=MAX_ITERATIONS,
            convergence_bound=CONVERGENCE_BOUND,
            n_subproblems=N_SUBPROBLEMS,
            parallel_dsp=False,  # Single-threaded for simplicity
        )
        
        # Wrap with time-budget enforcement
        solver_with_timeout = BendersSolverWithTimeout(solver, TIME_BUDGET_SECONDS)
        
        # Solve
        task_start = time.time()
        solved_milp, states, exit_reason = solver_with_timeout.solve(milp, Y_PREFIX, output_dir)
        task_duration = time.time() - task_start
        
        # Print summary
        final_state = states[-1] if states else None
        print(f"  ✓ Completed: {exit_reason} after {len(states)} iterations ({task_duration:.1f}s)")
        if final_state:
            gap = BendersSolverWithTimeout._compute_gap(final_state.lower_bound, final_state.upper_bound)
            total_qaoa_time = sum(s.runtimes.get("cut_selection_time", 0) or 0 for s in states)
            print(f"    Final bounds: LB={final_state.lower_bound:.2f}, UB={final_state.upper_bound:.2f}, gap={gap:.6f}")
            print(f"    Total QAOA time: {total_qaoa_time:.1f}s ({total_qaoa_time/task_duration*100:.1f}% of total)")
    
    except Exception as e:
        print(f"  ✗ Error: {e}")
        import traceback
        traceback.print_exc()
        # Write error to result file so we don't retry
        output_dir.mkdir(parents=True, exist_ok=True)
        with open(output_dir / "result.json", "w") as f:
            json.dump({"error": str(e), "exit_reason": "error"}, f)

print("\n" + "=" * 80)
print("All tasks completed!")
print(f"Results saved to {RESULTS_DIR}/")


[1/2] toy_0_p1
  Instance: toy_0
  QAOA depth (p): 1
  Output: experimental-data/results/toy-hqc-mcms/toy_0_p1


/zfsstore/user/valkcqde/documents/logistiqs/milp_engine/solvers/pulp_solver.py:46: UserWarning: No time limit set for the solver.
  warnings.warn("No time limit set for the solver.")
/zfsstore/user/valkcqde/documents/logistiqs/milp_engine/solvers/pulp_solver.py:51: UserWarning: No optimality gap set for the solver.
  warnings.warn("No optimality gap set for the solver.")


  Iteration 1: LB=-1.000000e+06, UB=inf, gap=inf, elapsed=79.6s


/zfsstore/user/valkcqde/documents/logistiqs/milp_engine/solvers/pulp_solver.py:46: UserWarning: No time limit set for the solver.
  warnings.warn("No time limit set for the solver.")
/zfsstore/user/valkcqde/documents/logistiqs/milp_engine/solvers/pulp_solver.py:51: UserWarning: No optimality gap set for the solver.
  warnings.warn("No optimality gap set for the solver.")


  Iteration 2: LB=-1.000000e+06, UB=inf, gap=inf, elapsed=133.5s


...

/zfsstore/user/valkcqde/documents/logistiqs/milp_engine/solvers/pulp_solver.py:46: UserWarning: No time limit set for the solver.
  warnings.warn("No time limit set for the solver.")
/zfsstore/user/valkcqde/documents/logistiqs/milp_engine/solvers/pulp_solver.py:51: UserWarning: No optimality gap set for the solver.
  warnings.warn("No optimality gap set for the solver.")


  Iteration 30: LB=1.848000e+01, UB=2.626000e+01, gap=0.296268, elapsed=1894.4s


/zfsstore/user/valkcqde/documents/logistiqs/milp_engine/solvers/pulp_solver.py:46: UserWarning: No time limit set for the solver.
  warnings.warn("No time limit set for the solver.")
/zfsstore/user/valkcqde/documents/logistiqs/milp_engine/solvers/pulp_solver.py:51: UserWarning: No optimality gap set for the solver.
  warnings.warn("No optimality gap set for the solver.")


  Iteration 31: LB=2.314000e+01, UB=2.626000e+01, gap=0.118812, elapsed=1952.5s
  ✓ Completed: converged after 32 iterations (1952.6s)
    Final bounds: LB=23.85, UB=23.85, gap=0.000000
    Total QAOA time: 1951.8s (100.0% of total)

All tasks completed!
Results saved to experimental-data/results/toy-hqc-mcms/


/zfsstore/user/valkcqde/documents/logistiqs/milp_engine/solvers/pulp_solver.py:46: UserWarning: No time limit set for the solver.
  warnings.warn("No time limit set for the solver.")
/zfsstore/user/valkcqde/documents/logistiqs/milp_engine/solvers/pulp_solver.py:51: UserWarning: No optimality gap set for the solver.
  warnings.warn("No optimality gap set for the solver.")


## Done

Results are in `experimental-data/results/toy-hqc-mcms/` with one subdirectory per task:
- `iterations.jsonl`: Per-iteration state (bounds, gaps, runtimes including QAOA time, cut counts)
- `result.json`: Final summary (exit reason, total time, final bounds)

These files are consumed by notebook 05 (generate figures) to produce Figure 3 from the paper.